In [1]:
# Keycloak Authorization Code Flow -- Remote Jupyter / Out-of-Band Edition
# -----------------------------------------------------------------
# No local server needed: click the login link, copy the redirect URL
# from your browser address bar, and paste it into the prompt below.

# ------------------------------------------------------------------
# CONFIG -- edit these values before running
# ------------------------------------------------------------------
KEYCLOAK_URL  = "https://lrllkeyclock.azurewebsites.net"
REALM         = "LowResourceLanguageDataTrust"
CLIENT_ID     = "LRLL-API-ID"
SCOPES        = "openid profile email"
CALLBACK_PORT = 8081   # must match the redirect URI registered in Keycloak
# ------------------------------------------------------------------


In [2]:
# -- Step 1: Get the login link ----------------------------------------
# Run this cell. Click the link, complete login in the browser.
# Your browser will redirect to a page that shows a connection error — that's expected.
# Copy the full URL from the browser address bar, then move to the next cell.

from ddplrll_reader.auth import login_oob_begin

_oob_session = login_oob_begin(
    KEYCLOAK_URL, REALM, CLIENT_ID, SCOPES, CALLBACK_PORT,
)


In [3]:
# -- Step 2: Paste the redirect URL and complete login -----------------
# Paste the full URL from your browser's address bar into REDIRECT_URL below,
# then run this cell.

from ddplrll_reader.auth import login_oob_complete

REDIRECT_URL = "http://127.0.0.1:8081/callback?state=B9Xpk7ISYkCU4iYLNU65_w&session_state=PxLk84PdI44lECQWj7aDVAOa&iss=https%3A%2F%2Flrllkeyclock.azurewebsites.net%2Frealms%2FLowResourceLanguageDataTrust&code=b982b324-71a7-646a-f9c3-79ff2d97213e.PxLk84PdI44lECQWj7aDVAOa.bd4d8a1f-50f6-416f-a2c9-e488aa3fcdb2"  # <-- paste here, e.g. "http://127.0.0.1:8081/callback?code=...&state=..."

login_oob_complete(_oob_session, REDIRECT_URL, verify_ssl=False)


[+] Authorization code captured (b982b324-71a...)
[*] Exchanging code for tokens ...

  TOKEN RESPONSE
  Token type  : Bearer
  Expires in  : 300s

  Access token:
  eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJlLXc4MEtuakhZV1M2bTNVcWtjUGV1...

  ID token    :
  eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJlLXc4MEtuakhZV1M2bTNVcWtjUGV1...

  Refresh token:
  eyJhbGciOiJIUzUxMiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJjYjM4ZmRmZS04MDhiLTRkZWQtYjI5...

  Tokens saved to: tokens.json

[+] Done. Run  ddplrll-reader run  to query the API.


{'access_token': 'eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICJlLXc4MEtuakhZV1M2bTNVcWtjUGV1TUc3YkZfOHBmanRXRUFjekQzNUhVIn0.eyJleHAiOjE3ODUzNTQ3MTAsImlhdCI6MTc4NTM1NDQxMCwiYXV0aF90aW1lIjoxNzg1MzU0MTg0LCJqdGkiOiJvbnJ0YWM6OGYxN2E1NDItYjMzYS0zYTI5LWI0Y2MtMzBlZjQzMDA0ODg0IiwiaXNzIjoiaHR0cHM6Ly9scmxsa2V5Y2xvY2suYXp1cmV3ZWJzaXRlcy5uZXQvcmVhbG1zL0xvd1Jlc291cmNlTGFuZ3VhZ2VEYXRhVHJ1c3QiLCJhdWQiOlsiTFJMTC1BUEktSUQiLCJhY2NvdW50Il0sInN1YiI6IjhhMWQ0ZDQxLWRiMTMtNGI4Mi1hNmY3LThiYjNlNDY0ZGNlNyIsInR5cCI6IkJlYXJlciIsImF6cCI6IkxSTEwtQVBJLUlEIiwic2lkIjoiUHhMazg0UGRJNDRsRUNRV2o3YURWQU9hIiwiYWNyIjoiMCIsImFsbG93ZWQtb3JpZ2lucyI6WyJodHRwOi8vMTI3LjAuMC4xIl0sInJlYWxtX2FjY2VzcyI6eyJyb2xlcyI6WyJvZmZsaW5lX2FjY2VzcyIsInVtYV9hdXRob3JpemF0aW9uIiwiZGVmYXVsdC1yb2xlcy1sb3dyZXNvdXJjZWxhbmd1YWdlZGF0YXRydXN0Il19LCJyZXNvdXJjZV9hY2Nlc3MiOnsiTFJMTC1BUEktSUQiOnsicm9sZXMiOlsiYXBpLXJlYWQiXX0sImFjY291bnQiOnsicm9sZXMiOlsibWFuYWdlLWFjY291bnQiLCJtYW5hZ2UtYWNjb3VudC1saW5rcyIsInZpZXctcHJvZmlsZSJdfX0sInNjb3BlIjoib3BlbmlkIGVtYWlsIH

---
## Query the API

The cells below use `DdplrllDatasetClient` (which automatically reads `tokens.json`) to query **text** and **sound** datasets. Adjust the filter parameters as needed.

In [4]:
# -- Query text datasets -----------------------------------------
#
# Filters (all optional): keyword, theme, author, year, limit
# Results: {TEXT_OUTPUT}/dataset.jsonld  and  {TEXT_OUTPUT}/files/

from ddplrll_reader import DdplrllDatasetClient, Settings, preview_jsonld

TEXT_OUTPUT = "./output/text"

settings = Settings(
    api_base_url="https://lrllapi.azurewebsites.net",
    output_dir=TEXT_OUTPUT,
    verify_ssl=False,
)
client = DdplrllDatasetClient(settings)

jsonld_path = await client.arun(
    keyword="malaria",
    # theme="HEALTH",
    # author="John Banda",
    limit=10,
    output_dir=TEXT_OUTPUT,
    download=True,
)

print(f"[+] JSON-LD saved to: {jsonld_path}")
preview_jsonld(jsonld_path)


[+] JSON-LD saved to: /Users/mozdag/Projects/lowreslanglibapireader/output/text/dataset.jsonld

Matched: 3  Returned: 3  Random sample: False

Dataset : output_extracts_nation_newspaper
Files   : 3

First file:
  Name    : Bungwe lilimbana ndi malungo
  URL     : /Users/mozdag/Projects/lowreslanglibapireader/output/text/files/file-output_extracts_nation_newspaper-5fa6de4b.md
  Author  : Patrick Ndawala
  Keywords: ['Malaria', 'NGO', 'Women', 'Collaboration', 'Health campaign']
  Language: ny
  Format  : text/markdown


In [ ]:
# -- Query sound datasets ----------------------------------------
#
# Filters (all optional): language, programme, keyword, year, limit
# Results: {SOUND_OUTPUT}/sound_dataset.jsonld  and  {SOUND_OUTPUT}/files/
# file_download_timeout caps per-file streaming time to avoid hangs.

from ddplrll_reader import DdplrllDatasetClient, Settings, preview_jsonld

SOUND_OUTPUT = "./output/sound"

settings = Settings(
    api_base_url="http://localhost:5000",
    output_dir=SOUND_OUTPUT,
    file_download_timeout=120.0,  # seconds; raise if files are large
)
client = DdplrllDatasetClient(settings)

sound_path = await client.arun_sound(
    language="ny",
    # programme="Radio Chichewa",
    # keyword="climate",
    year=2024,
    limit=10,
    output_dir=SOUND_OUTPUT,
    download=True,
)

print(f"[+] Sound JSON-LD saved to: {sound_path}")
preview_jsonld(sound_path)


CancelledError: 